# Audit và thống kê các data split

Notebook này **không tạo lại split**. Nó đọc `data/splits/train.csv`, `valid.csv`, `test.csv` và kiểm tra trước khi training.

Quy ước đang được audit:

- `image_id` là tên file đầy đủ, bao gồm extension;
- `image_path` có dạng `train/train/<filename>`;
- `mask_path` có dạng `train_gt/train_gt/<filename>`;
- cả hai path được resolve tương đối từ thư mục `data/`;
- `foreground_pixels` và `foreground_ratio` không lưu trong CSV mà được tính khi thống kê.

In [1]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

EXPECTED_ROWS = {"train": 700, "valid": 100, "test": 200}
EXPECTED_LABELED = {"labeled_1_4": 175, "labeled_1_2": 350}
SPLIT_NAMES = ("train", "valid", "test")

COMMON_COLUMNS = [
    "image_id", "image_path", "mask_path", "stratum",
    "has_non_neoplastic", "has_neoplastic",
    "width", "height", "background_pixels",
    "non_neo_pixels", "neo_pixels",
]
TRAIN_COLUMNS = COMMON_COLUMNS + ["labeled_1_4", "labeled_1_2"]


def find_project_root(start=Path.cwd()):
    for root in [start.resolve(), *start.resolve().parents]:
        if (root / "data" / "splits").is_dir():
            return root
    raise FileNotFoundError(
        "Không tìm thấy data/splits. Hãy chạy notebook từ project root "
        "hoặc một thư mục con."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
SPLIT_DIR = DATA_DIR / "splits"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("SPLIT_DIR    =", SPLIT_DIR)

PROJECT_ROOT = B:\_projects\poply-image-segmentation
DATA_DIR     = B:\_projects\poply-image-segmentation\data
SPLIT_DIR    = B:\_projects\poply-image-segmentation\data\splits


## 1. Load CSV và xem bảng dữ liệu

In [2]:
csv_paths = {name: SPLIT_DIR / f"{name}.csv" for name in SPLIT_NAMES}
missing_csvs = [str(path) for path in csv_paths.values() if not path.is_file()]
if missing_csvs:
    raise FileNotFoundError("Thiếu CSV:\n- " + "\n- ".join(missing_csvs))

splits = {name: pd.read_csv(path) for name, path in csv_paths.items()}

for df in splits.values():
    df.columns = df.columns.str.strip()
    for col in ("stratum", "image_id", "image_path", "mask_path"):
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()
    for col in (
        "labeled_1_4", "labeled_1_2",
        "has_non_neoplastic", "has_neoplastic",
        "width", "height", "background_pixels",
        "non_neo_pixels", "neo_pixels",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

for name, df in splits.items():
    display(Markdown(f"### `{name}.csv` — {len(df):,} dòng × {df.shape[1]} cột"))
    display(df.head(10))

### `train.csv` — 700 dòng × 13 cột

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels,labeled_1_4,labeled_1_2
0,0081835cf877e004e8bfb905b78a9139.jpeg,train/train/0081835cf877e004e8bfb905b78a9139.jpeg,train_gt/train_gt/0081835cf877e004e8bfb905b78a9139.jpeg,neo_only,0,1,1280,995,1252742,0,20858,0,1
1,00f9b6c911c4aa179344271fa54aa1b2.jpeg,train/train/00f9b6c911c4aa179344271fa54aa1b2.jpeg,train_gt/train_gt/00f9b6c911c4aa179344271fa54aa1b2.jpeg,neo_only,0,1,1280,959,1173318,0,54202,1,1
2,00fd197cd955fa095f978455cef3593c.jpeg,train/train/00fd197cd955fa095f978455cef3593c.jpeg,train_gt/train_gt/00fd197cd955fa095f978455cef3593c.jpeg,non_neo_only,1,0,1280,977,1143416,107144,0,0,0
3,01c13a3d810b000db55759486b8aa1b8.jpeg,train/train/01c13a3d810b000db55759486b8aa1b8.jpeg,train_gt/train_gt/01c13a3d810b000db55759486b8aa1b8.jpeg,non_neo_only,1,0,1280,995,1244543,29057,0,0,1
4,01e80947f9c02314fc58bdb26679bff5.jpeg,train/train/01e80947f9c02314fc58bdb26679bff5.jpeg,train_gt/train_gt/01e80947f9c02314fc58bdb26679bff5.jpeg,neo_only,0,1,1280,995,1161474,0,112126,0,1
5,02000f9b6c911c4aa179344271fa54aa.jpeg,train/train/02000f9b6c911c4aa179344271fa54aa.jpeg,train_gt/train_gt/02000f9b6c911c4aa179344271fa54aa.jpeg,neo_only,0,1,1280,959,1176594,0,50926,1,1
6,021bab9fc7330398846f67b5df7cdf3f.jpeg,train/train/021bab9fc7330398846f67b5df7cdf3f.jpeg,train_gt/train_gt/021bab9fc7330398846f67b5df7cdf3f.jpeg,neo_only,0,1,1280,959,989485,0,238035,1,1
7,02345d341253851bc866ebe4d18d5401.jpeg,train/train/02345d341253851bc866ebe4d18d5401.jpeg,train_gt/train_gt/02345d341253851bc866ebe4d18d5401.jpeg,neo_only,0,1,1280,995,1210610,0,62990,0,0
8,025ba0e3b6d0bf0b5633059989805234.jpeg,train/train/025ba0e3b6d0bf0b5633059989805234.jpeg,train_gt/train_gt/025ba0e3b6d0bf0b5633059989805234.jpeg,neo_only,0,1,1280,995,1231579,0,42021,0,0
9,02b296cc5cfd263f1f90be2879923502.jpeg,train/train/02b296cc5cfd263f1f90be2879923502.jpeg,train_gt/train_gt/02b296cc5cfd263f1f90be2879923502.jpeg,neo_only,0,1,1280,995,1250919,0,22681,0,1


### `valid.csv` — 100 dòng × 11 cột

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,0496b83ef461c2a337948a41964c1d4f.jpeg,train/train/0496b83ef461c2a337948a41964c1d4f.jpeg,train_gt/train_gt/0496b83ef461c2a337948a41964c1d4f.jpeg,non_neo_only,1,0,1280,995,1237881,35719,0
1,051fc90e85d064f787d4be168d951058.jpeg,train/train/051fc90e85d064f787d4be168d951058.jpeg,train_gt/train_gt/051fc90e85d064f787d4be168d951058.jpeg,neo_only,0,1,1280,977,1225389,0,25171
2,07907941a8490fd7df8e26031fbb5e52.jpeg,train/train/07907941a8490fd7df8e26031fbb5e52.jpeg,train_gt/train_gt/07907941a8490fd7df8e26031fbb5e52.jpeg,neo_only,0,1,1280,995,1256778,0,16822
3,0bbd4ce2a1405f7d507592b964b7955a.jpeg,train/train/0bbd4ce2a1405f7d507592b964b7955a.jpeg,train_gt/train_gt/0bbd4ce2a1405f7d507592b964b7955a.jpeg,neo_only,0,1,1280,995,1233760,0,39840
4,0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train/train/0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train_gt/train_gt/0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,non_neo_only,1,0,1280,977,1225582,24978,0
5,0f618d52893a23c466c16952b6692e6b.jpeg,train/train/0f618d52893a23c466c16952b6692e6b.jpeg,train_gt/train_gt/0f618d52893a23c466c16952b6692e6b.jpeg,non_neo_only,1,0,1280,995,1220433,53167,0
6,11f025ba0e3b6d0bf0b5633059989805.jpeg,train/train/11f025ba0e3b6d0bf0b5633059989805.jpeg,train_gt/train_gt/11f025ba0e3b6d0bf0b5633059989805.jpeg,non_neo_only,1,0,1280,995,1256381,17219,0
7,14fc58bdb26679bff55177a34fc01019.jpeg,train/train/14fc58bdb26679bff55177a34fc01019.jpeg,train_gt/train_gt/14fc58bdb26679bff55177a34fc01019.jpeg,neo_only,0,1,1280,995,1226438,0,47162
8,16d4ca6160127cd1d5ff99c267599fc4.jpeg,train/train/16d4ca6160127cd1d5ff99c267599fc4.jpeg,train_gt/train_gt/16d4ca6160127cd1d5ff99c267599fc4.jpeg,both,1,1,1280,995,1252564,6011,15025
9,177a34fc01019eec999fd84e679b05f9.jpeg,train/train/177a34fc01019eec999fd84e679b05f9.jpeg,train_gt/train_gt/177a34fc01019eec999fd84e679b05f9.jpeg,neo_only,0,1,1280,995,1035735,0,237865


### `test.csv` — 200 dòng × 11 cột

,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,00d827b8bcf15050fb9af0ef92846b60.jpeg,train/train/00d827b8bcf15050fb9af0ef92846b60.jpeg,train_gt/train_gt/00d827b8bcf15050fb9af0ef92846b60.jpeg,neo_only,0,1,1280,995,1197272,0,76328
1,01ff069910ea2abe3c84417fda801941.jpeg,train/train/01ff069910ea2abe3c84417fda801941.jpeg,train_gt/train_gt/01ff069910ea2abe3c84417fda801941.jpeg,neo_only,0,1,1280,995,1261623,0,11977
2,040781d54ffd6a8ee924dbd70f618d52.jpeg,train/train/040781d54ffd6a8ee924dbd70f618d52.jpeg,train_gt/train_gt/040781d54ffd6a8ee924dbd70f618d52.jpeg,neo_only,0,1,1280,995,1212723,0,60877
3,081835cf877e004e8bfb905b78a91391.jpeg,train/train/081835cf877e004e8bfb905b78a91391.jpeg,train_gt/train_gt/081835cf877e004e8bfb905b78a91391.jpeg,neo_only,0,1,1280,995,1237606,0,35994
4,099130d98309817287a8aef566999467.jpeg,train/train/099130d98309817287a8aef566999467.jpeg,train_gt/train_gt/099130d98309817287a8aef566999467.jpeg,neo_only,0,1,1280,995,1197251,0,76349
5,0a1230b118fa8625605da2023387fd56.jpeg,train/train/0a1230b118fa8625605da2023387fd56.jpeg,train_gt/train_gt/0a1230b118fa8625605da2023387fd56.jpeg,neo_only,0,1,1280,959,1104987,0,122533
6,0a22abd004c33abf3ae2136cd9dd77ae.jpeg,train/train/0a22abd004c33abf3ae2136cd9dd77ae.jpeg,train_gt/train_gt/0a22abd004c33abf3ae2136cd9dd77ae.jpeg,neo_only,0,1,1280,995,1251580,0,22020
7,0bf0b5633059989805234e222e19769f.jpeg,train/train/0bf0b5633059989805234e222e19769f.jpeg,train_gt/train_gt/0bf0b5633059989805234e222e19769f.jpeg,neo_only,0,1,1280,995,1208754,0,64846
8,0f1536ad3f6a5664c1711b62f15ec83b.jpeg,train/train/0f1536ad3f6a5664c1711b62f15ec83b.jpeg,train_gt/train_gt/0f1536ad3f6a5664c1711b62f15ec83b.jpeg,neo_only,0,1,1280,995,1181921,0,91679
9,117c6599c6dc55c83a46ce2be9800e6c.jpeg,train/train/117c6599c6dc55c83a46ce2be9800e6c.jpeg,train_gt/train_gt/117c6599c6dc55c83a46ce2be9800e6c.jpeg,non_neo_only,1,0,1280,995,1260021,13579,0


## 2. Schema, số dòng và giá trị thiếu

In [3]:
schema_rows = []
for name, df in splits.items():
    expected_columns = TRAIN_COLUMNS if name == "train" else COMMON_COLUMNS
    missing_columns = sorted(set(expected_columns) - set(df.columns))
    unexpected_columns = sorted(set(df.columns) - set(expected_columns))
    exact_schema = list(df.columns) == expected_columns

    schema_rows.append({
        "split": name,
        "rows": len(df),
        "expected_rows": EXPECTED_ROWS[name],
        "row_count_ok": len(df) == EXPECTED_ROWS[name],
        "columns": df.shape[1],
        "missing_columns": ", ".join(missing_columns) or "—",
        "unexpected_columns": ", ".join(unexpected_columns) or "—",
        "exact_schema_ok": exact_schema,
        "cells_with_na": int(df.isna().sum().sum()),
    })

schema_summary = pd.DataFrame(schema_rows).set_index("split")
display(schema_summary)

missing_values = pd.concat(
    {name: df.isna().sum() for name, df in splits.items()}, axis=1
).fillna(0).astype(int)

display(Markdown("### Số giá trị thiếu theo cột"))
if missing_values.values.sum():
    display(missing_values.loc[missing_values.sum(axis=1).gt(0)])
else:
    display(pd.DataFrame({"result": ["Không có NA"]}))

,rows,expected_rows,row_count_ok,columns,missing_columns,unexpected_columns,exact_schema_ok,cells_with_na
split,,,,,,,,
train,700,700,True,13,—,—,True,0
valid,100,100,True,11,—,—,True,0
test,200,200,True,11,—,—,True,0


### Số giá trị thiếu theo cột

,result
0,Không có NA


## 3. Duplicate trong split và leakage giữa các split

In [4]:
IDENTITY_COLUMNS = ('image_id', 'image_path', 'mask_path')
duplicate_rows = []
for name, df in splits.items():
    row = {'split': name, 'exact_duplicate_rows': int(df.duplicated(keep=False).sum())}
    for col in IDENTITY_COLUMNS:
        row[f'duplicate_{col}'] = (
            int(df[col].duplicated(keep=False).sum()) if col in df.columns else np.nan
        )
    duplicate_rows.append(row)
duplicate_summary = pd.DataFrame(duplicate_rows).set_index('split')
display(duplicate_summary)

leak_rows = []
for left, right in combinations(SPLIT_NAMES, 2):
    for col in IDENTITY_COLUMNS:
        if col in splits[left].columns and col in splits[right].columns:
            a = set(splits[left][col].dropna().astype(str))
            b = set(splits[right][col].dropna().astype(str))
            overlap = sorted(a & b)
            leak_rows.append({
                'split_pair': f'{left} ↔ {right}', 'key': col,
                'overlap_count': len(overlap),
                'examples': ', '.join(overlap[:5]) or '—',
            })
leak_summary = pd.DataFrame(leak_rows)
display(Markdown('### Giao nhau giữa các split'))
display(leak_summary)

,exact_duplicate_rows,duplicate_image_id,duplicate_image_path,duplicate_mask_path
split,,,,
train,0,0,0,0
valid,0,0,0,0
test,0,0,0,0


### Giao nhau giữa các split

,split_pair,key,overlap_count,examples
0,train ↔ valid,image_id,0,—
1,train ↔ valid,image_path,0,—
2,train ↔ valid,mask_path,0,—
3,train ↔ test,image_id,0,—
4,train ↔ test,image_path,0,—
5,train ↔ test,mask_path,0,—
6,valid ↔ test,image_id,0,—
7,valid ↔ test,image_path,0,—
8,valid ↔ test,mask_path,0,—


## 4. Kiểm tra đường dẫn image/mask

In [5]:
def resolve_dataset_path(value):
    """CSV path luôn tương đối từ DATA_DIR, không phải PROJECT_ROOT."""
    if pd.isna(value) or not str(value).strip():
        return None

    relative_path = Path(str(value))
    if relative_path.is_absolute():
        return None

    return (DATA_DIR / relative_path).resolve()


path_rows = []
missing_path_details = []
mapping_rows = []

for name, df in splits.items():
    for col, expected_prefix in (
        ("image_path", ("train", "train")),
        ("mask_path", ("train_gt", "train_gt")),
    ):
        resolved = df[col].map(resolve_dataset_path)
        exists = resolved.map(lambda path: path is not None and path.is_file())
        prefix_ok = df[col].map(
            lambda value: (
                not Path(str(value)).is_absolute()
                and Path(str(value)).parts[:2] == expected_prefix
            )
        )

        path_rows.append({
            "split": name,
            "path_column": col,
            "checked": len(df),
            "missing": int((~exists).sum()),
            "invalid_format": int((~prefix_ok).sum()),
            "all_exist": bool(exists.all()),
            "format_ok": bool(prefix_ok.all()),
        })

        bad = df.loc[
            ~(exists & prefix_ok),
            ["image_id", col],
        ].copy()
        if not bad.empty:
            bad.insert(0, "split", name)
            bad.insert(1, "path_column", col)
            missing_path_details.append(bad.head(20))

    mapping_ok = (
        df["image_path"].map(lambda value: Path(str(value)).name).eq(df["image_id"])
        & df["mask_path"].map(lambda value: Path(str(value)).name).eq(df["image_id"])
    )
    mapping_rows.append({
        "split": name,
        "checked": len(df),
        "mapping_violations": int((~mapping_ok).sum()),
        "mapping_ok": bool(mapping_ok.all()),
    })

path_summary = pd.DataFrame(path_rows)
mapping_summary = pd.DataFrame(mapping_rows).set_index("split")

display(path_summary)
display(Markdown("### Mapping `image_id = image filename = mask filename`"))
display(mapping_summary)

if missing_path_details:
    display(Markdown("### Ví dụ path sai hoặc không tồn tại"))
    display(pd.concat(missing_path_details, ignore_index=True))

,split,path_column,checked,missing,invalid_format,all_exist,format_ok
0,train,image_path,700,0,0,True,True
1,train,mask_path,700,0,0,True,True
2,valid,image_path,100,0,0,True,True
3,valid,mask_path,100,0,0,True,True
4,test,image_path,200,0,0,True,True
5,test,mask_path,200,0,0,True,True


### Mapping `image_id = image filename = mask filename`

,checked,mapping_violations,mapping_ok
split,,,
train,700,0,True
valid,100,0,True
test,200,0,True


## 5. Phân bố `stratum`

In [6]:
stratum_counts = pd.concat(
    {name: df['stratum'].value_counts(dropna=False) for name, df in splits.items() if 'stratum' in df},
    axis=1
).fillna(0).astype(int)
stratum_counts.columns.name = 'split'
stratum_pct = stratum_counts.div(stratum_counts.sum(axis=0), axis=1).mul(100).round(2)
display(Markdown('### Số ảnh'))
display(stratum_counts)
display(Markdown('### Tỷ lệ (%)'))
display(stratum_pct)

### Số ảnh

split,train,valid,test
stratum,,,
neo_only,486,69,139
non_neo_only,180,26,51
both,34,5,10


### Tỷ lệ (%)

split,train,valid,test
stratum,,,
neo_only,69.43,69.0,69.5
non_neo_only,25.71,26.0,25.5
both,4.86,5.0,5.0


## 6. Labeled 1/4 và 1/2; kiểm tra subset lồng nhau

In [7]:
train = splits['train']
label_rows = []
for col, expected in EXPECTED_LABELED.items():
    if col not in train.columns:
        label_rows.append({'setting': col, 'labeled': np.nan, 'unlabeled': np.nan, 'expected_labeled': expected, 'count_ok': False, 'binary_ok': False})
        continue
    values = train[col]
    binary_ok = bool(values.dropna().isin([0, 1]).all() and values.notna().all())
    labeled = int(values.eq(1).sum())
    label_rows.append({
        'setting': col, 'labeled': labeled, 'unlabeled': int(values.eq(0).sum()),
        'expected_labeled': expected, 'count_ok': labeled == expected, 'binary_ok': binary_ok,
    })
label_summary = pd.DataFrame(label_rows).set_index('setting')
display(label_summary)

label_distribution_parts = []
if 'stratum' in train.columns:
    for col in EXPECTED_LABELED:
        if col in train.columns:
            part = (train.assign(status=np.where(train[col].eq(1), 'labeled', 'unlabeled'))
                    .groupby(['status', 'stratum'], dropna=False).size()
                    .rename('images').reset_index())
            part.insert(0, 'setting', col)
            label_distribution_parts.append(part)
label_distribution = pd.concat(label_distribution_parts, ignore_index=True) if label_distribution_parts else pd.DataFrame()
display(Markdown('### Phân bố labeled/unlabeled theo stratum'))
display(label_distribution)

if {'labeled_1_4', 'labeled_1_2'} <= set(train.columns):
    nesting_violations = train['labeled_1_4'].eq(1) & ~train['labeled_1_2'].eq(1)
    nested_subset_ok = not nesting_violations.any()
    print(f'Nested subset 1/4 ⊂ 1/2: {nested_subset_ok} | violations: {int(nesting_violations.sum())}')
    if nesting_violations.any():
        display(train.loc[nesting_violations].head(20))
else:
    nested_subset_ok = False
    print('Không thể kiểm tra nesting vì thiếu cột labeled.')

,labeled,unlabeled,expected_labeled,count_ok,binary_ok
setting,,,,,
labeled_1_4,175,525,175,True,True
labeled_1_2,350,350,350,True,True


### Phân bố labeled/unlabeled theo stratum

,setting,status,stratum,images
0,labeled_1_4,labeled,both,8
1,labeled_1_4,labeled,neo_only,122
2,labeled_1_4,labeled,non_neo_only,45
3,labeled_1_4,unlabeled,both,26
4,labeled_1_4,unlabeled,neo_only,364
5,labeled_1_4,unlabeled,non_neo_only,135
6,labeled_1_2,labeled,both,17
7,labeled_1_2,labeled,neo_only,243
8,labeled_1_2,labeled,non_neo_only,90
9,labeled_1_2,unlabeled,both,17


Nested subset 1/4 ⊂ 1/2: True | violations: 0


## 7. Thống kê pixel và class

In [8]:
pixel_rows = []
pixel_validity_rows = []

for name, df in splits.items():
    background = df["background_pixels"]
    non_neo = df["non_neo_pixels"]
    neo = df["neo_pixels"]
    total_from_shape = df["width"] * df["height"]
    total_from_classes = background + non_neo + neo

    non_neo_total = int(non_neo.sum())
    neo_total = int(neo.sum())
    foreground_total = non_neo_total + neo_total

    pixel_rows.append({
        "split": name,
        "non_neo_pixels": non_neo_total,
        "neo_pixels": neo_total,
        "foreground_pixels": foreground_total,
        "non_neo_%_of_foreground": 100 * non_neo_total / foreground_total,
        "neo_%_of_foreground": 100 * neo_total / foreground_total,
        "images_with_non_neo": int(non_neo.gt(0).sum()),
        "images_with_neo": int(neo.gt(0).sum()),
    })

    pixel_validity_rows.append({
        "split": name,
        "negative_pixel_counts": int(
            background.lt(0).sum() + non_neo.lt(0).sum() + neo.lt(0).sum()
        ),
        "class_sum_mismatches": int(total_from_classes.ne(total_from_shape).sum()),
        "non_neo_flag_mismatches": int(
            df["has_non_neoplastic"].ne(non_neo.gt(0).astype(int)).sum()
        ),
        "neo_flag_mismatches": int(
            df["has_neoplastic"].ne(neo.gt(0).astype(int)).sum()
        ),
    })

pixel_summary = pd.DataFrame(pixel_rows).set_index("split")
pixel_validity = pd.DataFrame(pixel_validity_rows).set_index("split")

display(pixel_summary.style.format({
    "non_neo_pixels": "{:,.0f}",
    "neo_pixels": "{:,.0f}",
    "foreground_pixels": "{:,.0f}",
    "non_neo_%_of_foreground": "{:.2f}",
    "neo_%_of_foreground": "{:.2f}",
}))

display(Markdown("### Kiểm tra pixel count và cờ class"))
display(pixel_validity)

,non_neo_pixels,neo_pixels,foreground_pixels,non_neo_%_of_foreground,neo_%_of_foreground,images_with_non_neo,images_with_neo
split,,,,,,,
train,"5,655,991","40,030,832","45,686,823",12.38,87.62,214,520
valid,"1,269,472","4,744,460","6,013,932",21.11,78.89,31,74
test,"1,395,803","12,690,861","14,086,664",9.91,90.09,61,149


### Kiểm tra pixel count và cờ class

,negative_pixel_counts,class_sum_mismatches,non_neo_flag_mismatches,neo_flag_mismatches
split,,,,
train,0,0,0,0
valid,0,0,0,0
test,0,0,0,0


## 8. Foreground-ratio summaries

In [9]:
# foreground_ratio được suy ra, không yêu cầu tồn tại trong CSV.
foreground_ratios = {}
ratio_parts = []
ratio_validity = []

for name, df in splits.items():
    total_pixels = df["width"] * df["height"]
    foreground_pixels = df["non_neo_pixels"] + df["neo_pixels"]
    ratio = foreground_pixels / total_pixels
    foreground_ratios[name] = ratio

    desc = ratio.describe(percentiles=[.05, .25, .5, .75, .95]).to_frame().T
    desc.insert(0, "split", name)
    ratio_parts.append(desc)
    ratio_validity.append({
        "split": name,
        "source": "derived_from_pixel_counts",
        "missing": int(ratio.isna().sum()),
        "outside_[0,1]": int((ratio.lt(0) | ratio.gt(1)).sum()),
    })

foreground_summary = pd.concat(ratio_parts, ignore_index=True).set_index("split")
display(foreground_summary.style.format("{:.4f}", na_rep="N/A"))
display(pd.DataFrame(ratio_validity).set_index("split"))

ratio_bins = [-np.inf, 0, .01, .05, .10, .25, .50, 1, np.inf]
ratio_labels = [
    "≤0", "(0,1%]", "(1,5%]", "(5,10%]",
    "(10,25%]", "(25,50%]", "(50,100%]", ">100%",
]
binned = {
    name: pd.cut(ratio, ratio_bins, labels=ratio_labels).value_counts(sort=False)
    for name, ratio in foreground_ratios.items()
}

display(Markdown("### Số ảnh theo khoảng foreground ratio"))
display(pd.DataFrame(binned).fillna(0).astype(int))

,count,mean,std,min,5%,25%,50%,75%,95%,max
split,,,,,,,,,,
train,700.0000,0.0519,0.0536,0.0014,0.0065,0.0180,0.0356,0.0652,0.1705,0.4069
valid,100.0000,0.0479,0.0620,0.0043,0.0088,0.0186,0.0301,0.0521,0.1484,0.4411
test,200.0000,0.0561,0.0684,0.0022,0.0063,0.0155,0.0347,0.0698,0.1729,0.4581


,source,missing,"outside_[0,1]"
split,,,
train,derived_from_pixel_counts,0,0
valid,derived_from_pixel_counts,0,0
test,derived_from_pixel_counts,0,0


### Số ảnh theo khoảng foreground ratio

,train,valid,test
≤0,0,0,0
"(0,1%]",70,8,28
"(1,5%]",392,65,106
"(5,10%]",148,18,34
"(10,25%]",82,7,27
"(25,50%]",8,2,5
"(50,100%]",0,0,0
>100%,0,0,0


## 9. Bảng tổng hợp cuối trước khi training

In [10]:
compact_rows = []
for name, df in splits.items():
    strata = df["stratum"].value_counts()
    path_part = path_summary[path_summary["split"].eq(name)]
    compact_rows.append({
        "split": name,
        "rows": len(df),
        "neo_only": int(strata.get("neo_only", 0)),
        "non_neo_only": int(strata.get("non_neo_only", 0)),
        "both": int(strata.get("both", 0)),
        "background_only": int(strata.get("background_only", 0)),
        "duplicate_identity_rows": int(
            duplicate_summary.loc[name]
            .drop(labels="exact_duplicate_rows")
            .fillna(0)
            .max()
        ),
        "missing_files": int(path_part["missing"].sum()),
        "invalid_path_format": int(path_part["invalid_format"].sum()),
        "mapping_violations": int(mapping_summary.loc[name, "mapping_violations"]),
        "foreground_ratio_mean": float(foreground_ratios[name].mean()),
    })

compact_summary = pd.DataFrame(compact_rows).set_index("split")
display(compact_summary.style.format({"foreground_ratio_mean": "{:.4f}"}))

checks = []


def add_check(check, passed, detail):
    checks.append({
        "status": "PASS" if bool(passed) else "FAIL",
        "check": check,
        "detail": detail,
    })


add_check(
    "Đủ số dòng 700/100/200",
    schema_summary["row_count_ok"].all(),
    str(schema_summary["rows"].to_dict()),
)
add_check(
    "Schema chính xác",
    schema_summary["exact_schema_ok"].all(),
    "train=13 cột; valid/test=11 cột",
)
add_check(
    "Không duplicate identity trong từng split",
    duplicate_summary.drop(columns="exact_duplicate_rows").fillna(0).to_numpy().max() == 0,
    "image_id/image_path/mask_path",
)
add_check(
    "Không leakage giữa split",
    leak_summary["overlap_count"].sum() == 0,
    f"tổng overlap = {int(leak_summary['overlap_count'].sum())}",
)
add_check(
    "Tất cả image/mask path tồn tại",
    path_summary["all_exist"].all(),
    f"thiếu {int(path_summary['missing'].sum())} file",
)
add_check(
    "Path đúng định dạng tương đối từ data/",
    path_summary["format_ok"].all(),
    "train/train/... và train_gt/train_gt/...",
)
add_check(
    "image_id khớp chính xác tên image/mask",
    mapping_summary["mapping_ok"].all(),
    f"vi phạm {int(mapping_summary['mapping_violations'].sum())}",
)
add_check(
    "Pixel count khớp width × height",
    pixel_validity["class_sum_mismatches"].sum() == 0,
    f"mismatch {int(pixel_validity['class_sum_mismatches'].sum())}",
)
add_check(
    "Đủ 175 ảnh labeled 1/4",
    bool(label_summary.loc["labeled_1_4", "count_ok"]),
    str(label_summary.loc["labeled_1_4", "labeled"]),
)
add_check(
    "Đủ 350 ảnh labeled 1/2",
    bool(label_summary.loc["labeled_1_2", "count_ok"]),
    str(label_summary.loc["labeled_1_2", "labeled"]),
)
add_check(
    "Labeled 1/4 là subset của 1/2",
    nested_subset_ok,
    "không có 1/4=1 nhưng 1/2!=1",
)
add_check(
    "Foreground ratio suy ra nằm trong [0,1]",
    sum(row["outside_[0,1]"] for row in ratio_validity) == 0,
    "tính từ pixel counts và width × height",
)

validation_checklist = pd.DataFrame(checks)
display(validation_checklist)

if validation_checklist["status"].eq("PASS").all():
    display(Markdown("## ✅ Split đã vượt qua toàn bộ kiểm tra tự động."))
else:
    display(Markdown("## ⚠️ Split còn kiểm tra FAIL — xem chi tiết trước khi training."))

,rows,neo_only,non_neo_only,both,background_only,duplicate_identity_rows,missing_files,invalid_path_format,mapping_violations,foreground_ratio_mean
split,,,,,,,,,,
train,700,486,180,34,0,0,0,0,0,0.0519
valid,100,69,26,5,0,0,0,0,0,0.0479
test,200,139,51,10,0,0,0,0,0,0.0561


,status,check,detail
0,PASS,Đủ số dòng 700/100/200,"{'train': 700, 'valid': 100, 'test': 200}"
1,PASS,Schema chính xác,train=13 cột; valid/test=11 cột
2,PASS,Không duplicate identity trong từng split,image_id/image_path/mask_path
3,PASS,Không leakage giữa split,tổng overlap = 0
4,PASS,Tất cả image/mask path tồn tại,thiếu 0 file
5,PASS,Path đúng định dạng tương đối từ data/,train/train/... và train_gt/train_gt/...
6,PASS,image_id khớp chính xác tên image/mask,vi phạm 0
7,PASS,Pixel count khớp width × height,mismatch 0
8,PASS,Đủ 175 ảnh labeled 1/4,175
9,PASS,Đủ 350 ảnh labeled 1/2,350


## ✅ Split đã vượt qua toàn bộ kiểm tra tự động.

### Ghi chú diễn giải

- `test.csv` là internal test có ground truth, không phải competition test không có mask.
- `labeled_1_4 = 1` phải kéo theo `labeled_1_2 = 1`.
- CSV path được ghép với `DATA_DIR`, ví dụ `DATA_DIR / "train/train/abc.jpeg"`.
- Foreground statistics được suy ra từ `non_neo_pixels`, `neo_pixels`, `width`, `height`; không cần lưu các cột foreground trong CSV.
- Notebook dùng metadata pixel đã lưu trong CSV và không mở lại mask, nhờ đó audit nhanh và độc lập với notebook tạo split.